# TensorFlow — Google's End-to-End Machine Learning Platform

---

## What Is TensorFlow?

TensorFlow is an open-source machine learning framework developed by Google Brain, released in 2015. It is designed to take a model from **research to production** across every platform — servers, mobile devices, browsers, and edge devices.

Where PyTorch is the favorite of **researchers**, TensorFlow has historically been the favorite of **production engineers**, because of its rich deployment tooling:
- **TensorFlow Serving** — serve models as REST/gRPC APIs at scale
- **TensorFlow Lite** — run models on Android/iOS with tiny footprint
- **TensorFlow.js** — run models directly in the browser
- **TensorFlow Extended (TFX)** — full ML pipeline orchestration

### Real-World Analogy

Think of building a car (neural network). PyTorch is like a **custom workshop** — total flexibility, great for prototyping new designs, but you assemble every piece manually. TensorFlow is like a **car factory assembly line** — slightly more structured, but once the pipeline is set up it can produce thousands of identical cars (model predictions) efficiently, with built-in quality control (monitoring, versioning, A/B testing).

---

## TF1 vs TF2 — A Critical Distinction

TensorFlow had a major redesign between v1 and v2 (released 2019):

| Feature | TF1 | TF2 |
|---|---|---|
| Execution | Static graph (define then run) | **Eager execution** (runs immediately, like NumPy) |
| API | `tf.Session`, `tf.placeholder` | **Keras-first** (`model.fit()`) |
| Debugging | Painful (can't `print()` tensors) | Natural Python debugging |
| Default | Off | **On by default** |

**If you find old TF1 code online, ignore the session/placeholder pattern — TF2 is completely different and much simpler.**

---

## Prerequisites

- Python basics (classes, functions, loops)
- NumPy fundamentals
- Basic ML/DL concepts (train/test split, loss functions, gradient descent)
- Reading the PyTorch notebook first is helpful but not required

---

## Table of Contents

1. Installation & Setup
2. TensorFlow Tensors — The Core Data Type
3. Variables & Gradient Tape — TF's Autograd
4. The Keras API — Building Models
5. The Training API — `model.compile()` + `model.fit()`
6. Custom Training Loops with GradientTape
7. `tf.data` — Efficient Data Pipelines
8. Callbacks — Control Training Behavior
9. Saving & Loading Models
10. `tf.function` — Speed Up with Graph Execution
11. Mini Project — Customer Churn Prediction
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://www.tensorflow.org/api_docs/python/  
**Tutorials:** https://www.tensorflow.org/tutorials  
**GitHub:** https://github.com/tensorflow/tensorflow  
**YouTube — TensorFlow Beginner (TF official):** https://www.youtube.com/watch?v=tPYj3fFJGjk  
**YouTube — MIT 6.S191 Deep Learning:** https://www.youtube.com/watch?v=ErnWZxJovaM  

## 1. Installation & Setup

```bash
pip install tensorflow          # CPU (works on any machine)
pip install tensorflow[and-cuda]  # GPU support on Linux
```

On Apple Silicon Macs, use:
```bash
pip install tensorflow-macos tensorflow-metal
```

In [ ]:
tensorflowtry:
    import tensorflow as tf
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    import warnings; warnings.filterwarnings("ignore")
    print(f"TensorFlow {tf.__version__} ready | GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
except ImportError:
    raise SystemExit("Run: pip install tensorflow scikit-learn pandas numpy matplotlib")


## 2. TensorFlow Tensors — The Core Data Type

Like PyTorch tensors and NumPy arrays, TF tensors are multi-dimensional arrays. Key difference: **TF tensors are immutable** (you can't change their value in place). For mutable values use `tf.Variable`.

In [ ]:
# Creating tensors
t_const  = tf.constant([1.0, 2.0, 3.0])                    # from list
t_zeros  = tf.zeros([3, 4])                                  # 3x4 zeros
t_ones   = tf.ones([2, 3])                                   # 2x3 ones
t_rand   = tf.random.uniform([2, 3], minval=0, maxval=1)    # uniform random
t_randn  = tf.random.normal([2, 3], mean=0, stddev=1)       # normal random
t_range  = tf.range(0, 10, delta=2)                          # [0, 2, 4, 6, 8]
t_from_np = tf.constant(np.array([1, 2, 3], dtype=np.float32))  # from NumPy

print("Shape:",  t_zeros.shape)    # (3, 4)
print("Dtype:",  t_const.dtype)    # float32
print("Device:", t_const.device)   # CPU or GPU path

# Operations
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[5.0, 6.0], [7.0, 8.0]])

print("\nAdd:    \n", (a + b).numpy())
print("Matmul: \n", tf.matmul(a, b).numpy())
print("Sum:    ",    tf.reduce_sum(a).numpy())
print("Mean:   ",    tf.reduce_mean(a).numpy())

# Convert to NumPy (eager mode makes this trivial)
np_array = a.numpy()
print("\nTo NumPy:", np_array)

In [ ]:
# Shape manipulation
x = tf.range(12, dtype=tf.float32)
print("Original:",   x.shape)                         # (12,)
print("Reshaped:",   tf.reshape(x, [3, 4]).shape)     # (3, 4)
print("Expanded:",   tf.expand_dims(x, 0).shape)      # (1, 12)
print("Squeezed:",   tf.squeeze(tf.expand_dims(x, 0)).shape)  # (12,)
print("Transposed:", tf.transpose(tf.reshape(x, [3,4])).shape)  # (4, 3)

# Indexing and slicing (same as NumPy)
mat = tf.constant([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=tf.float32)
print("\nRow 0:",      mat[0].numpy())
print("Col 1:",        mat[:, 1].numpy())
print("Submatrix:",    mat[0:2, 1:3].numpy())

## 3. Variables & Gradient Tape — TF's Autograd

- **`tf.Variable`**: a mutable tensor — used for model weights that need to be updated during training
- **`tf.GradientTape`**: records operations so gradients can be computed (TF's equivalent of PyTorch's autograd)

Inside a `with tf.GradientTape() as tape:` block, TF records every operation. After the block, `tape.gradient(loss, variables)` computes the derivatives.

In [ ]:
# GradientTape basics
x = tf.Variable(4.0)   # mutable variable, gradient will be computed for this

with tf.GradientTape() as tape:
    y = x**2 + 3*x + 2   # y = x^2 + 3x + 2

dy_dx = tape.gradient(y, x)  # dy/dx = 2x + 3 = 11 at x=4
print(f"y at x=4:    {y.numpy():.1f}")      # 30
print(f"dy/dx at x=4: {dy_dx.numpy():.1f}") # 11

# Manual gradient descent (to understand what Keras does under the hood)
w = tf.Variable(2.0)   # model weight to learn
x_data = tf.constant([1.0, 2.0, 3.0, 4.0])
y_true  = tf.constant([3.0, 6.0, 9.0, 12.0])  # y = 3x

print("\n--- Manual Gradient Descent (learning y = 3x) ---")
lr = 0.05
for step in range(6):
    with tf.GradientTape() as tape:
        y_pred = w * x_data
        loss   = tf.reduce_mean((y_pred - y_true) ** 2)  # MSE

    grad = tape.gradient(loss, w)   # dLoss/dw
    w.assign_sub(lr * grad)          # w = w - lr * grad  (.assign_sub modifies Variable in place)
    print(f"Step {step+1}: w={w.numpy():.4f}, loss={loss.numpy():.4f}")

print(f"\nFinal w ≈ {w.numpy():.4f}  (target: 3.0)")

## 4. The Keras API — Building Models

Keras is TensorFlow's high-level API (fully integrated since TF2). It provides three ways to build models, from simple to fully flexible:

1. **Sequential API** — layer stack, one input → one output, no branching
2. **Functional API** — explicit data flow, supports multi-input/output, branching, shared layers
3. **Model subclassing** — full Python control (like PyTorch's `nn.Module`)

In [ ]:
# ==================================================
# Method 1: Sequential API (quick and simple)
# ==================================================
model_seq = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # binary classification output
], name='sequential_model')

model_seq.summary()  # prints architecture + parameter counts

In [ ]:
# ==================================================
# Method 2: Functional API (multi-input capable)
# ==================================================
# Example: a model that takes both numeric features AND categorical embeddings

# Numeric branch
numeric_input = tf.keras.Input(shape=(8,), name='numeric_features')
x1 = tf.keras.layers.Dense(32, activation='relu')(numeric_input)
x1 = tf.keras.layers.BatchNormalization()(x1)

# Categorical branch (integer IDs → embedding vectors)
cat_input = tf.keras.Input(shape=(1,), dtype='int32', name='category_id')
x2 = tf.keras.layers.Embedding(input_dim=50, output_dim=8)(cat_input)  # 50 categories → 8-dim
x2 = tf.keras.layers.Flatten()(x2)

# Merge branches
merged = tf.keras.layers.Concatenate()([x1, x2])
merged = tf.keras.layers.Dense(64, activation='relu')(merged)
merged = tf.keras.layers.Dropout(0.3)(merged)
output = tf.keras.layers.Dense(1, activation='sigmoid')(merged)

model_func = tf.keras.Model(
    inputs=[numeric_input, cat_input],
    outputs=output,
    name='functional_model'
)
model_func.summary()

In [ ]:
# ==================================================
# Method 3: Model subclassing (most flexible)
# ==================================================

class ResidualBlock(tf.keras.layers.Layer):
    """A single residual block: output = relu(x + Dense(x))."""
    def __init__(self, units):
        super().__init__()
        self.dense = tf.keras.layers.Dense(units)
        self.bn    = tf.keras.layers.BatchNormalization()

    def call(self, x, training=False):
        return tf.nn.relu(x + self.bn(self.dense(x), training=training))


class ResNet(tf.keras.Model):
    def __init__(self, hidden_size, num_blocks, num_classes):
        super().__init__()
        self.input_proj = tf.keras.layers.Dense(hidden_size, activation='relu')
        self.blocks     = [ResidualBlock(hidden_size) for _ in range(num_blocks)]
        self.dropout    = tf.keras.layers.Dropout(0.3)
        self.output_layer = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, x, training=False):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.dropout(x, training=training)
        return self.output_layer(x)

resnet = ResNet(hidden_size=64, num_blocks=3, num_classes=5)
sample = tf.random.normal([8, 20])  # batch of 8, 20 features
out    = resnet(sample, training=False)
print(f"ResNet output shape: {out.shape}")  # (8, 5)
print(f"Probabilities sum: {tf.reduce_sum(out[0]).numpy():.4f}")  # should be ~1.0

## 5. The Training API — `model.compile()` + `model.fit()`

Keras's biggest advantage: a **4-line training workflow** that handles the training loop, validation, metrics, and callbacks.

```python
model.compile(optimizer, loss, metrics)  # configure training
history = model.fit(X_train, y_train,    # train
                    epochs=...,
                    validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)           # test
model.predict(X_new)                     # inference
```

In [ ]:
# ==================================================
# Full example: binary classification on synthetic data
# ==================================================

np.random.seed(42)
n = 2000
X_syn = np.random.randn(n, 15).astype(np.float32)
# Target depends on first 5 features
p = 1 / (1 + np.exp(-(X_syn[:, 0] * 2 + X_syn[:, 1] - X_syn[:, 2] + X_syn[:, 3] * 0.5)))
y_syn = (np.random.rand(n) < p).astype(np.float32)

X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape}, Val: {X_val.shape}, Test: {X_te.shape}")

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(15,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile: specify optimizer, loss, and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Train
history = model.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=0  # suppress per-epoch output
)

# Evaluate
results = model.evaluate(X_te, y_te, verbose=0)
metric_names = ['loss', 'accuracy', 'auc', 'precision', 'recall']
print("\n--- Test Results ---")
for name, val in zip(metric_names, results):
    print(f"  {name:12s}: {val:.4f}")

In [ ]:
# Plot training curves (history object records all metrics)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history.history['loss'],     label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Binary Cross-Entropy Loss')
axes[0].legend()

# AUC
axes[1].plot(history.history['auc'],     label='Train AUC')
axes[1].plot(history.history['val_auc'], label='Val AUC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_title('ROC-AUC Score')
axes[1].legend()

plt.tight_layout()
plt.show()

# Inference
probs = model.predict(X_te[:5], verbose=0)
print("\nPredicted probabilities (first 5 test samples):")
for i, (p, y) in enumerate(zip(probs.flatten(), y_te[:5])):
    print(f"  Sample {i}: prob={p:.3f}, true label={int(y)}")

## 6. Custom Training Loops with GradientTape

When you need more control than `model.fit()` provides (e.g., training a GAN, custom gradient clipping, curriculum learning), you write your own training loop using `GradientTape`.

In [ ]:
# Custom training loop — gives you full control

custom_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(15,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer  = tf.keras.optimizers.Adam(0.001)
loss_fn    = tf.keras.losses.BinaryCrossentropy()
train_acc  = tf.keras.metrics.BinaryAccuracy()
val_acc    = tf.keras.metrics.BinaryAccuracy()

# Convert to tf.data.Dataset for the custom loop
batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr)).shuffle(1000).batch(batch_size)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size)

@tf.function  # compile to a fast TF graph (see section 10)
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        predictions = custom_model(x_batch, training=True)
        loss = loss_fn(y_batch, predictions)
    # Compute and apply gradients
    grads = tape.gradient(loss, custom_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, custom_model.trainable_variables))
    train_acc.update_state(y_batch, predictions)
    return loss

@tf.function
def val_step(x_batch, y_batch):
    predictions = custom_model(x_batch, training=False)
    val_acc.update_state(y_batch, predictions)

EPOCHS = 20
for epoch in range(EPOCHS):
    train_acc.reset_state(); val_acc.reset_state()
    total_loss = 0.0; steps = 0

    for xb, yb in train_ds:
        loss = train_step(xb, yb)
        total_loss += loss; steps += 1

    for xb, yb in val_ds:
        val_step(xb, yb)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
              f"Loss: {total_loss/steps:.4f} | "
              f"Train Acc: {train_acc.result().numpy():.4f} | "
              f"Val Acc: {val_acc.result().numpy():.4f}")

## 7. `tf.data` — Efficient Data Pipelines

`tf.data.Dataset` is TensorFlow's high-performance data pipeline. It handles loading, preprocessing, batching, shuffling, and prefetching — often the **bottleneck** in deep learning is data loading, not computation.

Key methods:
- `.from_tensor_slices(tensors)` — create from arrays
- `.map(fn)` — apply a function to each element (e.g., augmentation)
- `.filter(fn)` — remove elements matching a condition
- `.shuffle(buffer_size)` — randomly shuffle
- `.batch(n)` — group into batches
- `.prefetch(tf.data.AUTOTUNE)` — load next batch while GPU processes current batch
- `.cache()` — cache dataset in memory after first epoch

In [ ]:
# Build an optimized data pipeline

BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE

def normalize(x, y):
    """Normalize features (this would typically be done on the dataset stats)."""
    return tf.cast(x, tf.float32), tf.cast(y, tf.float32)

def add_noise(x, y):
    """Data augmentation: add small Gaussian noise to features."""
    noisy_x = x + tf.random.normal(tf.shape(x), stddev=0.01)
    return noisy_x, y

train_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .map(normalize, num_parallel_calls=AUTOTUNE)  # preprocess in parallel
    .map(add_noise, num_parallel_calls=AUTOTUNE)   # augmentation
    .shuffle(buffer_size=500)                       # shuffle with buffer
    .batch(BATCH_SIZE, drop_remainder=True)         # batch
    .cache()                                        # cache in memory
    .prefetch(AUTOTUNE)                             # prefetch next batch
)

val_pipeline = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .map(normalize, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(AUTOTUNE)
)

# Check one batch
for xb, yb in train_pipeline.take(1):
    print(f"Batch X shape: {xb.shape}")
    print(f"Batch y shape: {yb.shape}")

# Can pass tf.data.Dataset directly to model.fit()
model.fit(
    train_pipeline,
    epochs=5,
    validation_data=val_pipeline,
    verbose=1
)

## 8. Callbacks — Control Training Behavior

Callbacks are functions that run at specific points during training (start of epoch, end of batch, etc.). They let you:
- Stop training early when it stops improving
- Save the best model checkpoint
- Reduce learning rate when stuck
- Log metrics to TensorBoard

In [ ]:
import tempfile, os

save_dir = tempfile.mkdtemp()

callbacks = [
    # Stop training if val_loss doesn't improve for 10 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,  # revert to best checkpoint on stop
        verbose=1
    ),

    # Save the best model automatically
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(save_dir, 'best_model.keras'),
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=0
    ),

    # Halve LR if val_loss doesn't improve for 5 epochs
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    # Log to TensorBoard (view with: tensorboard --logdir /tmp/logs)
    tf.keras.callbacks.TensorBoard(
        log_dir=os.path.join(save_dir, 'logs'),
        histogram_freq=1
    )
]

# Rebuild fresh model
cb_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(15,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
cb_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

cb_history = cb_model.fit(
    X_tr, y_tr,
    epochs=100,       # will stop early via EarlyStopping
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=0
)

print(f"\nTraining stopped at epoch: {len(cb_history.history['loss'])}")
test_res = cb_model.evaluate(X_te, y_te, verbose=0)
print(f"Test AUC: {test_res[2]:.4f}")

## 9. Saving & Loading Models

In [ ]:
# Method 1: SavedModel format (recommended — includes architecture + weights + computation)
saved_model_path = os.path.join(save_dir, 'my_model')
model.save(saved_model_path)
loaded = tf.keras.models.load_model(saved_model_path)
print("SavedModel loaded successfully")

# Method 2: Keras native format (.keras file)
keras_path = os.path.join(save_dir, 'my_model.keras')
model.save(keras_path)
loaded_keras = tf.keras.models.load_model(keras_path)
print("Keras format loaded successfully")

# Method 3: Weights only (needs same architecture to load)
weights_path = os.path.join(save_dir, 'weights.weights.h5')
model.save_weights(weights_path)
model.load_weights(weights_path)
print("Weights loaded successfully")

# Verify predictions are identical
orig_pred   = model.predict(X_te[:3], verbose=0).flatten()
loaded_pred = loaded.predict(X_te[:3], verbose=0).flatten()
print(f"\nOriginal predictions:  {orig_pred.round(4)}")
print(f"Loaded predictions:    {loaded_pred.round(4)}")
print(f"Max difference: {np.max(np.abs(orig_pred - loaded_pred)):.2e}")

## 10. `tf.function` — Speed Up with Graph Execution

By default, TF2 runs in **eager mode** (immediate execution, great for debugging). The `@tf.function` decorator **compiles a function into a TF computation graph**, which can be 2-10x faster for repeated calls.

**When to use:** Wrap your training/inference steps for production speed. Don't wrap code you're actively debugging.

In [ ]:
import time

x_bench = tf.random.normal([1000, 15])

# Eager function (no decorator)
def predict_eager(model, x):
    return model(x, training=False)

# Graph-compiled function
@tf.function
def predict_fast(model, x):
    return model(x, training=False)

# Warm up (first call traces the graph)
_ = predict_eager(model, x_bench)
_ = predict_fast(model, x_bench)

# Benchmark
N_CALLS = 500

t0 = time.time()
for _ in range(N_CALLS):
    predict_eager(model, x_bench)
eager_time = time.time() - t0

t0 = time.time()
for _ in range(N_CALLS):
    predict_fast(model, x_bench)
graph_time = time.time() - t0

print(f"Eager mode:  {eager_time:.3f}s  ({N_CALLS} calls)")
print(f"Graph mode:  {graph_time:.3f}s  ({N_CALLS} calls)")
print(f"Speedup:     {eager_time/graph_time:.2f}x")

## 11. Mini Project — Customer Churn Prediction

### The Business Problem

A telecom company wants to predict which customers will cancel their subscription (churn). They want to call at-risk customers with retention offers before they leave.

**Goal:** Build a production-quality TensorFlow pipeline with proper preprocessing, class imbalance handling, callbacks, and evaluation.

In [ ]:
# ==================================================
# STEP 1: Generate realistic churn dataset
# ==================================================

np.random.seed(2024)
N = 5000

tenure_months  = np.random.exponential(24, N).clip(1, 120).astype(int)
monthly_charge = np.random.uniform(20, 120, N).round(2)
num_products   = np.random.choice([1, 2, 3, 4, 5], N, p=[0.3, 0.3, 0.2, 0.15, 0.05])
support_calls  = np.random.poisson(2, N).clip(0, 10)
payment_delay  = np.random.exponential(3, N).clip(0, 30).astype(int)
usage_gb       = np.random.gamma(2, 5, N).round(2)
contract_type  = np.random.choice([0, 1, 2], N, p=[0.5, 0.3, 0.2])  # 0=monthly, 1=1yr, 2=2yr
has_fiber      = np.random.binomial(1, 0.6, N)
has_security   = np.random.binomial(1, 0.4, N)
has_backup     = np.random.binomial(1, 0.3, N)

# Churn probability (realistic: monthly contract + high support calls + low tenure → churn)
p_churn = (
    0.05
    + (contract_type == 0) * 0.20      # monthly customers churn more
    + (support_calls >= 4) * 0.15      # frustrated customers churn
    + (tenure_months < 12) * 0.10      # new customers churn more
    + (payment_delay > 10) * 0.10      # payment issues → churn
    - (num_products >= 3) * 0.05       # more products = more sticky
    - (contract_type == 2) * 0.10      # 2-year contracts: lower churn
)
p_churn = np.clip(p_churn, 0.02, 0.90)
churn = (np.random.rand(N) < p_churn).astype(np.float32)

features = np.column_stack([
    tenure_months, monthly_charge, num_products, support_calls,
    payment_delay, usage_gb, contract_type, has_fiber, has_security, has_backup
]).astype(np.float32)

feature_names = ['tenure', 'monthly_charge', 'num_products', 'support_calls',
                 'payment_delay', 'usage_gb', 'contract_type', 'fiber', 'security', 'backup']

print(f"Dataset: {features.shape}, Churn rate: {churn.mean():.1%}")
print(f"Imbalance: {(churn==0).sum()} not churned, {(churn==1).sum()} churned")

In [ ]:
# ==================================================
# STEP 2: Preprocessing and data splitting
# ==================================================

X_trv, X_te, y_trv, y_te = train_test_split(features, churn, test_size=0.15, random_state=42, stratify=churn)
X_tr, X_val, y_tr, y_val = train_test_split(X_trv, y_trv, test_size=0.15, random_state=42, stratify=y_trv)

# Normalize (StandardScaler on train, apply to val/test)
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_te_s  = scaler.transform(X_te).astype(np.float32)

# Handle class imbalance with class weights
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
class_weight = {0: 1.0, 1: neg / pos}
print(f"Class weights → 0: {class_weight[0]:.1f}, 1: {class_weight[1]:.2f}")

# ==================================================
# STEP 3: Build model
# ==================================================

def build_churn_model(input_dim, dropout_rate=0.3):
    inputs  = tf.keras.Input(shape=(input_dim,), name='features')
    x = tf.keras.layers.Dense(128, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    output = tf.keras.layers.Dense(1, activation='sigmoid', name='churn_prob')(x)
    return tf.keras.Model(inputs, output, name='ChurnPredictor')

churn_model = build_churn_model(input_dim=10)
churn_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

churn_model.summary()

In [ ]:
# ==================================================
# STEP 4: Train with callbacks
# ==================================================

churn_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=15, mode='max',
                                      restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=7, min_lr=1e-6, verbose=0)
]

churn_history = churn_model.fit(
    X_tr_s, y_tr,
    epochs=100,
    batch_size=128,
    validation_data=(X_val_s, y_val),
    class_weight=class_weight,
    callbacks=churn_callbacks,
    verbose=0
)

print(f"Stopped at epoch: {len(churn_history.history['loss'])}")

test_results = churn_model.evaluate(X_te_s, y_te, verbose=0)
print("\n=== Test Results ===")
for name, val in zip(['Loss','Accuracy','AUC','Precision','Recall'], test_results):
    print(f"  {name:10s}: {val:.4f}")

In [ ]:
# ==================================================
# STEP 5: Business impact analysis
# ==================================================

y_proba = churn_model.predict(X_te_s, verbose=0).flatten()

# Use threshold 0.4 (lower than 0.5 to catch more churners — recall over precision)
threshold  = 0.4
y_pred_bin = (y_proba >= threshold).astype(int)

TP = ((y_pred_bin == 1) & (y_te == 1)).sum()
FP = ((y_pred_bin == 1) & (y_te == 0)).sum()
FN = ((y_pred_bin == 0) & (y_te == 1)).sum()
TN = ((y_pred_bin == 0) & (y_te == 0)).sum()

# Business assumptions
avg_monthly_revenue = 65    # $65/month per customer
retention_offer_cost = 20   # $20 cost to call + offer
conversion_rate      = 0.35 # 35% of retained customers accept and stay
avg_ltv_months       = 24   # expected lifetime after retention: 24 months

revenue_saved   = TP * conversion_rate * avg_ltv_months * avg_monthly_revenue
campaign_cost   = (TP + FP) * retention_offer_cost
net_roi         = revenue_saved - campaign_cost
missed_revenue  = FN * avg_ltv_months * avg_monthly_revenue

print("=" * 45)
print("   BUSINESS IMPACT ANALYSIS")
print("=" * 45)
print(f"  True Positives  (caught churners): {TP}")
print(f"  False Positives (wasted calls)   : {FP}")
print(f"  False Negatives (missed churners): {FN}")
print(f"  True Negatives  (correct stays)  : {TN}")
print(f"  Precision: {TP/(TP+FP):.3f}  Recall: {TP/(TP+FN):.3f}")
print("=" * 45)
print(f"  Revenue saved:  ${revenue_saved:>10,.0f}")
print(f"  Campaign cost: -${campaign_cost:>10,.0f}")
print(f"  Net ROI:        ${net_roi:>10,.0f}")
print(f"  Missed revenue: ${missed_revenue:>10,.0f}")
print("=" * 45)

## 12. Common Pitfalls

### Pitfall 1: Normalizing with Test Statistics

Always fit your scaler on **training data only**, then transform val/test. Fitting on the full dataset leaks test information into training.

```python
# WRONG
scaler.fit_transform(all_data)

# CORRECT
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)  # use train statistics!
```

### Pitfall 2: Wrong Loss Function for the Task

| Task | Output Activation | Loss |
|---|---|---|
| Binary classification | `sigmoid` | `binary_crossentropy` |
| Multi-class (one-hot y) | `softmax` | `categorical_crossentropy` |
| Multi-class (integer y) | `softmax` | `sparse_categorical_crossentropy` |
| Regression | Linear (none) | `mse` or `mae` |

### Pitfall 3: `training=True/False` in Custom Layers

When building custom `tf.keras.Model` subclasses, always pass the `training` argument to layers that have different training/inference behavior (Dropout, BatchNorm).

```python
def call(self, x, training=False):  # ← default False
    x = self.dropout(x, training=training)  # pass it through!
```

### Pitfall 4: Ignoring Class Imbalance

Always compute `class_weight` when your dataset is imbalanced (>70/30 split) and pass it to `model.fit(class_weight=...)`. Or use `tf.keras.metrics.AUC` instead of accuracy — accuracy is misleading on imbalanced data.

### Pitfall 5: Not Using `@tf.function` in Production

Eager mode is slow for repeated inference. Always wrap your predict function with `@tf.function` in production.

## 13. Interview Q&A

---

**Q1: What is eager execution in TensorFlow? How is it different from graph execution?**

> **Eager execution** (default in TF2) means operations are evaluated immediately as Python calls them — just like NumPy. `tf.constant([1,2]) + tf.constant([3,4])` immediately returns `[4, 6]`. This makes debugging natural (you can `print()` tensors). **Graph execution** (TF1 style, or when using `@tf.function`) first builds a computation graph (a description of what to compute), then runs it later. Graph mode is faster for repeated execution (JIT compilation, operation fusion) but harder to debug.

---

**Q2: What does `@tf.function` do and when should you use it?**

> `@tf.function` is a decorator that converts a Python function into a TensorFlow graph. The first call traces the function (runs Python code to build the graph), and subsequent calls reuse the compiled graph without Python overhead — typically 2-10x faster. Use it for training/inference loops that run many times. Don't use it during debugging, and be careful with Python side effects (like `print()`) inside `@tf.function` — they only run during tracing.

---

**Q3: What are the three Keras model-building APIs?**

> 1. **Sequential API**: `tf.keras.Sequential([layer1, layer2, ...])` — simple stack, one input, one output, no branching
> 2. **Functional API**: explicit `Input` → layer chaining → `Model(inputs, outputs)` — supports multi-input/output, branching, shared layers
> 3. **Model subclassing**: `class MyModel(tf.keras.Model)` with `__init__` (define layers) and `call` (define computation) — most flexible, supports any architecture

---

**Q4: What does `GradientTape` do and when do you use it?**

> `tf.GradientTape` records operations inside its `with` block. After the block, `tape.gradient(loss, variables)` computes derivatives using the chain rule. Use it when you need a **custom training loop** (instead of `model.fit()`) — for example, when training GANs (two models with separate gradient updates), implementing custom regularization, or performing gradient surgery.

---

**Q5: What is `tf.data.Dataset` and why is it better than feeding NumPy arrays directly?**

> `tf.data.Dataset` builds a **lazy data pipeline** that:
> - Loads and preprocesses data **only when needed** (memory efficient for large datasets)
> - Uses **parallel processing** (`num_parallel_calls=AUTOTUNE`) for map operations
> - **Prefetches** the next batch to the GPU while the current batch is being processed
> - **Caches** the dataset in memory after the first epoch
> Feeding NumPy arrays directly can make the GPU wait idle while the CPU loads and preprocesses data.

---

**Q6: What callbacks would you use in a production training run?**

> 1. `EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)` — stop when overfitting, restore best model
> 2. `ModelCheckpoint(save_best_only=True)` — save the best model to disk
> 3. `ReduceLROnPlateau(factor=0.5, patience=5)` — reduce learning rate when stuck
> 4. `TensorBoard(log_dir=...)` — visualize metrics, weights, and computation graph
> 5. (Optional) Custom callback for alerting/logging to external systems

## 14. Resources

### Official
- **TensorFlow Documentation:** https://www.tensorflow.org/api_docs/python/
- **TF Tutorials:** https://www.tensorflow.org/tutorials
- **Keras Documentation:** https://keras.io/
- **TFX (TensorFlow Extended):** https://www.tensorflow.org/tfx

### Videos
- **MIT 6.S191 — Deep Learning (free course):** https://www.youtube.com/watch?v=ErnWZxJovaM
- **TF official tutorial playlist:** https://www.youtube.com/c/TensorFlow
- **Coursera Deep Learning Specialization:** https://www.coursera.org/specializations/deep-learning

### Books
- **Hands-On ML with Scikit-Learn, Keras & TF (Aurélien Géron):** https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/
- **Deep Learning with Python (François Chollet, Keras creator):** https://www.manning.com/books/deep-learning-with-python-second-edition

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **Tensors** | Immutable n-d arrays; use `tf.Variable` for mutable (trainable) values |
| **Eager execution** | Default in TF2; operations run immediately like NumPy |
| **GradientTape** | `with tape: ... tape.gradient(loss, vars)` — TF's autograd mechanism |
| **Sequential API** | `tf.keras.Sequential([layers])` — quick linear stacks |
| **Functional API** | `Input → layers → Model(in, out)` — multi-input/output, branching |
| **compile + fit** | 4-line training: compile → fit → evaluate → predict |
| **tf.data** | Efficient pipelines: `.map().shuffle().batch().cache().prefetch()` |
| **Callbacks** | EarlyStopping + ModelCheckpoint + ReduceLROnPlateau for production |
| **@tf.function** | 2-10x speedup by compiling functions to TF graphs |

### What's Next

- **Keras** — the standalone Keras (now Keras 3), with multi-backend support (TensorFlow, JAX, PyTorch)
- **JAX** — Google's NumPy-like library with JIT compilation and automatic differentiation
